In [1]:
import json

import logomaker as lm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyliftover
import seaborn as sns
from pyfaidx import Fasta
from scipy.stats import chi2_contingency

from analysis_functions import (
                        calculate_chi2_p_values,
                        filter_and_convert_to_list,
                        check_ref,
                        get_codon_info,
                        get_context
                    )

## 1. Separate variants by pathogenicity value

Create dataframes for pathogenic/benign variants based on frequency.

* **pathogenic**  
 Cutoff in AC < 2. Additionally, intersect with the options in ClinVar and add pathogenic/likely pathogenic variants that are missing in GnomAD v.4, but are in ClinVar.

* **benign**  
 AC cut-off >= 2 (according to recent ACGS guidelines, BS2 criterion). In this case, we may have many autosomal recessive variants left, so let’s remove them. To do this, compare the resulting dataframe with benign ClinVar variants and remove all intersections with registered P/LP variants.

In [2]:
nmd_undergo_df = pd.read_csv("data/lof_final_with_loeuf_pext_nmd_undergo.csv")

In [3]:
nmd_undergo_df.shape

(21116, 31)

In [4]:
pat_nmd_undergo = nmd_undergo_df.query('AC < 2')

In [5]:
pat_nmd_undergo

,CHROM,POS,ID,REF,ALT,AC,Consequence,IMPACT,SYMBOL,Gene,...,FLAGS,VARIANT_CLASS,CANONICAL,LoF,LoF_filter,LoF_flags,LoF_info,LOEUF,pext,NMD_escape
0,chr1,1825438,rs867938404,G,A,1,stop_gained,HIGH,GNB1,ENSG00000078369,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.0156402737047898,0.145,0.966547,NO
1,chr1,2024966,NaN,C,A,1,stop_gained,HIGH,GABRD,ENSG00000187730,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.0684326710816777,0.245,1.000000,NO
2,chr1,2024966,NaN,C,G,1,stop_gained,HIGH,GABRD,ENSG00000187730,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.0684326710816777,0.245,1.000000,NO
3,chr1,2024993,NaN,G,A,1,stop_gained,HIGH,GABRD,ENSG00000187730,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.0883002207505519,0.245,1.000000,NO
4,chr1,2025553,NaN,G,A,1,stop_gained,HIGH,GABRD,ENSG00000187730,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.209713024282561,0.245,1.000000,NO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21107,chr22,50720328,NaN,C,A,1,stop_gained,HIGH,SHANK3,ENSG00000251322,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.434494195688226,0.123,1.000000,NO
21108,chr22,50720377,NaN,C,A,1,stop_gained,HIGH,SHANK3,ENSG00000251322,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.444651741293532,0.123,1.000000,NO
21109,chr22,50720479,NaN,T,A,1,stop_gained,HIGH,SHANK3,ENSG00000251322,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.465796019900498,0.123,1.000000,NO
21111,chr22,50720939,NaN,G,T,1,stop_gained,HIGH,SHANK3,ENSG00000251322,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.561152570480929,0.123,1.000000,NO


In [6]:
ben_nmd_undergo = nmd_undergo_df.query('AC >= 2')

In [7]:
ben_nmd_undergo

,CHROM,POS,ID,REF,ALT,AC,Consequence,IMPACT,SYMBOL,Gene,...,FLAGS,VARIANT_CLASS,CANONICAL,LoF,LoF_filter,LoF_flags,LoF_info,LOEUF,pext,NMD_escape
5,chr1,2025660,NaN,C,A,2,stop_gained,HIGH,GABRD,ENSG00000187730,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.288447387785136,0.245,1.000000,NO
8,chr1,2028164,rs1441225021,C,G,3,stop_gained,HIGH,GABRD,ENSG00000187730,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.414275202354673,0.245,1.000000,NO
11,chr1,2029236,rs1405824159,C,T,5,stop_gained,HIGH,GABRD,ENSG00000187730,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.601177336276674,0.245,1.000000,NO
12,chr1,2228818,rs1225111926,C,T,2,stop_gained,HIGH,SKI,ENSG00000157933,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.0237768632830361,0.194,0.950938,NO
25,chr1,2509920,rs1424097228,G,A,3,stop_gained,HIGH,PANK4,ENSG00000157881,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.882859603789836,0.304,0.694764,NO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21105,chr22,50720238,NaN,C,A,3,stop_gained,HIGH,SHANK3,ENSG00000251322,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.415837479270315,0.123,1.000000,NO
21110,chr22,50720534,NaN,G,T,2,stop_gained,HIGH,SHANK3,ENSG00000251322,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.477197346600332,0.123,1.000000,NO
21113,chr22,50721521,NaN,G,T,5,stop_gained,HIGH,SHANK3,ENSG00000251322,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.681799336650083,0.123,1.000000,NO
21114,chr22,50721995,NaN,G,T,2,stop_gained,HIGH,SHANK3,ENSG00000251322,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.780058043117745,0.123,1.000000,NO


In [8]:
ben_nmd_undergo.columns

Index(['CHROM', 'POS', 'ID', 'REF', 'ALT', 'AC', 'Consequence', 'IMPACT',
       'SYMBOL', 'Gene', 'Feature', 'BIOTYPE', 'EXON', 'INTRON',
       'cDNA_position', 'CDS_position', 'Protein_position', 'Amino_acids',
       'Codons', 'ALLELE_NUM', 'STRAND', 'FLAGS', 'VARIANT_CLASS', 'CANONICAL',
       'LoF', 'LoF_filter', 'LoF_flags', 'LoF_info', 'LOEUF', 'pext',
       'NMD_escape'],
      dtype='object')

In [9]:
pat_nmd_undergo['AC'].value_counts()  # 1    1669

AC
1    15868
Name: count, dtype: int64

In [10]:
ben_nmd_undergo['AC'].value_counts()  # from 2 to 79

AC
2       2794
3       1067
4        541
5        300
6        156
7        102
8         65
9         47
10        34
11        24
12        23
13        15
16        12
14        10
19         6
15         6
24         4
22         4
26         3
32         3
21         3
20         3
25         2
23         2
17         2
18         1
111        1
36         1
229        1
118        1
28         1
89         1
71         1
1104       1
74         1
51         1
233        1
44         1
57         1
108        1
284        1
35         1
53         1
55         1
50         1
Name: count, dtype: int64

In [12]:
ben_nmd_undergo['AC'].describe()

count    5248.000000
mean        3.833079
std        16.886237
min         2.000000
25%         2.000000
50%         2.000000
75%         4.000000
max      1104.000000
Name: AC, dtype: float64

### Remove all pathogenic Clinvar variants from benign dataframe

Merge `clinvar_nmd_undergo_df` and `ben_nmd_undergo` dataframes, remove all intersections by `CHROM`, `POS`, `REF`, `ALT`, and then remove the remainder of `clinvar_nmd_undergo_df` (i.e. remove all rows that do not have an empty `CLNSIG` column).

In [13]:
clinvar_nmd_undergo_df = pd.read_csv("data/clinvar_nmd_undergo_df.csv")
# clinvar_nmd_undergo = clinvar_nmd_undergo_df.rename(columns={'Feature': 'Canonical_transcript'})

In [14]:
clinvar_nmd_undergo_df

,CHROM,POS,ID,REF,ALT,CLNREVSTAT,CLNSIG,CLNVC,GENEINFO,MC,...,CDS_position,Protein_position,Amino_acids,Codons,STRAND,FLAGS,CANONICAL,LOEUF,pext,NMD_escape
0,chr1,1790454,986264,G,A,"criteria_provided,_single_submitter",Pathogenic,single_nucleotide_variant,GNB1:2782,SO:0001587|nonsense,...,640.0,214.0,R/*,Cga/Tga,-1.0,NaN,YES,0.145,0.951698,NO
1,chr1,1815771,3281766,C,T,"criteria_provided,_single_submitter",Pathogenic,single_nucleotide_variant,GNB1:2782,"SO:0001587|nonsense,SO:0001627|intron_variant",...,188.0,63.0,W/*,tGg/tAg,-1.0,NaN,YES,0.145,0.966547,NO
2,chr1,1825399,1433729,G,A,"criteria_provided,_single_submitter",Pathogenic,single_nucleotide_variant,GNB1:2782,"SO:0001587|nonsense,SO:0001623|5_prime_UTR_var...",...,55.0,19.0,R/*,Cga/Tga,-1.0,NaN,YES,0.145,0.966547,NO
3,chr1,3385247,1029845,C,A,"criteria_provided,_single_submitter",Pathogenic,single_nucleotide_variant,PRDM16:63976,SO:0001587|nonsense,...,534.0,178.0,C/*,tgC/tgA,1.0,NaN,YES,0.187,0.848889,NO
4,chr1,3385272,3340802,C,T,"criteria_provided,_single_submitter",Pathogenic,single_nucleotide_variant,PRDM16:63976,SO:0001587|nonsense,...,559.0,187.0,Q/*,Cag/Tag,1.0,NaN,YES,0.187,0.848889,NO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7097,chr22,41176400,183678,C,T,"criteria_provided,_multiple_submitters,_no_con...",Pathogenic,single_nucleotide_variant,EP300:2033,SO:0001587|nonsense,...,4933.0,1645.0,R/*,Cga/Tga,1.0,NaN,YES,0.099,1.000000,NO
7098,chr22,50447442,1301645,G,C,"criteria_provided,_single_submitter",Likely_pathogenic,single_nucleotide_variant,SBF1:6305,SO:0001587|nonsense,...,5463.0,1821.0,Y/*,taC/taG,-1.0,NaN,YES,0.265,0.771752,NO
7099,chr22,50448330,3644969,G,A,"criteria_provided,_single_submitter",Pathogenic,single_nucleotide_variant,SBF1:6305,SO:0001587|nonsense,...,5266.0,1756.0,Q/*,Cag/Tag,-1.0,NaN,YES,0.265,0.771752,NO
7100,chr22,50448592,2018999,C,T,"criteria_provided,_single_submitter",Pathogenic,single_nucleotide_variant,SBF1:6305,SO:0001587|nonsense,...,5102.0,1701.0,W/*,tGg/tAg,-1.0,NaN,YES,0.265,0.771752,NO


In [15]:
clinvar_nmd_undergo_df.columns

Index(['CHROM', 'POS', 'ID', 'REF', 'ALT', 'CLNREVSTAT', 'CLNSIG', 'CLNVC',
       'GENEINFO', 'MC', 'Consequence', 'SYMBOL', 'Gene', 'Feature_type',
       'Feature', 'BIOTYPE', 'cDNA_position', 'CDS_position',
       'Protein_position', 'Amino_acids', 'Codons', 'STRAND', 'FLAGS',
       'CANONICAL', 'LOEUF', 'pext', 'NMD_escape'],
      dtype='object')

In [16]:
merged_clinvar_and_ben = pd.concat([ben_nmd_undergo, clinvar_nmd_undergo_df], ignore_index=True)
merged_clinvar_and_ben

,CHROM,POS,ID,REF,ALT,AC,Consequence,IMPACT,SYMBOL,Gene,...,LoF_info,LOEUF,pext,NMD_escape,CLNREVSTAT,CLNSIG,CLNVC,GENEINFO,MC,Feature_type
0,chr1,2025660,NaN,C,A,2.0,stop_gained,HIGH,GABRD,ENSG00000187730,...,PERCENTILE:0.288447387785136,0.245,1.000000,NO,NaN,NaN,NaN,NaN,NaN,NaN
1,chr1,2028164,rs1441225021,C,G,3.0,stop_gained,HIGH,GABRD,ENSG00000187730,...,PERCENTILE:0.414275202354673,0.245,1.000000,NO,NaN,NaN,NaN,NaN,NaN,NaN
2,chr1,2029236,rs1405824159,C,T,5.0,stop_gained,HIGH,GABRD,ENSG00000187730,...,PERCENTILE:0.601177336276674,0.245,1.000000,NO,NaN,NaN,NaN,NaN,NaN,NaN
3,chr1,2228818,rs1225111926,C,T,2.0,stop_gained,HIGH,SKI,ENSG00000157933,...,PERCENTILE:0.0237768632830361,0.194,0.950938,NO,NaN,NaN,NaN,NaN,NaN,NaN
4,chr1,2509920,rs1424097228,G,A,3.0,stop_gained,HIGH,PANK4,ENSG00000157881,...,PERCENTILE:0.882859603789836,0.304,0.694764,NO,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12345,chr22,41176400,183678,C,T,NaN,stop_gained,NaN,EP300,ENSG00000100393,...,NaN,0.099,1.000000,NO,"criteria_provided,_multiple_submitters,_no_con...",Pathogenic,single_nucleotide_variant,EP300:2033,SO:0001587|nonsense,Transcript
12346,chr22,50447442,1301645,G,C,NaN,stop_gained,NaN,SBF1,ENSG00000100241,...,NaN,0.265,0.771752,NO,"criteria_provided,_single_submitter",Likely_pathogenic,single_nucleotide_variant,SBF1:6305,SO:0001587|nonsense,Transcript
12347,chr22,50448330,3644969,G,A,NaN,stop_gained,NaN,SBF1,ENSG00000100241,...,NaN,0.265,0.771752,NO,"criteria_provided,_single_submitter",Pathogenic,single_nucleotide_variant,SBF1:6305,SO:0001587|nonsense,Transcript
12348,chr22,50448592,2018999,C,T,NaN,stop_gained,NaN,SBF1,ENSG00000100241,...,NaN,0.265,0.771752,NO,"criteria_provided,_single_submitter",Pathogenic,single_nucleotide_variant,SBF1:6305,SO:0001587|nonsense,Transcript


In [17]:
merged_clinvar_and_ben.columns

Index(['CHROM', 'POS', 'ID', 'REF', 'ALT', 'AC', 'Consequence', 'IMPACT',
       'SYMBOL', 'Gene', 'Feature', 'BIOTYPE', 'EXON', 'INTRON',
       'cDNA_position', 'CDS_position', 'Protein_position', 'Amino_acids',
       'Codons', 'ALLELE_NUM', 'STRAND', 'FLAGS', 'VARIANT_CLASS', 'CANONICAL',
       'LoF', 'LoF_filter', 'LoF_flags', 'LoF_info', 'LOEUF', 'pext',
       'NMD_escape', 'CLNREVSTAT', 'CLNSIG', 'CLNVC', 'GENEINFO', 'MC',
       'Feature_type'],
      dtype='object')

In [18]:
# remove duplicates
ben_nmd_undergo_filtered = merged_clinvar_and_ben.drop_duplicates(subset=['CHROM', 'POS', 'REF', 'ALT'], keep=False)

In [19]:
# remove the Clinvar df
ben_nmd_undergo_filtered = ben_nmd_undergo_filtered[~ben_nmd_undergo_filtered['CLNSIG'].notna()]

In [20]:
ben_nmd_undergo_filtered.columns

Index(['CHROM', 'POS', 'ID', 'REF', 'ALT', 'AC', 'Consequence', 'IMPACT',
       'SYMBOL', 'Gene', 'Feature', 'BIOTYPE', 'EXON', 'INTRON',
       'cDNA_position', 'CDS_position', 'Protein_position', 'Amino_acids',
       'Codons', 'ALLELE_NUM', 'STRAND', 'FLAGS', 'VARIANT_CLASS', 'CANONICAL',
       'LoF', 'LoF_filter', 'LoF_flags', 'LoF_info', 'LOEUF', 'pext',
       'NMD_escape', 'CLNREVSTAT', 'CLNSIG', 'CLNVC', 'GENEINFO', 'MC',
       'Feature_type'],
      dtype='object')

In [21]:
pat_nmd_undergo.columns

Index(['CHROM', 'POS', 'ID', 'REF', 'ALT', 'AC', 'Consequence', 'IMPACT',
       'SYMBOL', 'Gene', 'Feature', 'BIOTYPE', 'EXON', 'INTRON',
       'cDNA_position', 'CDS_position', 'Protein_position', 'Amino_acids',
       'Codons', 'ALLELE_NUM', 'STRAND', 'FLAGS', 'VARIANT_CLASS', 'CANONICAL',
       'LoF', 'LoF_filter', 'LoF_flags', 'LoF_info', 'LOEUF', 'pext',
       'NMD_escape'],
      dtype='object')

Remove unnecessary columns left after Clinvar.

In [22]:
# ben_nmd_undergo_filtered = ben_nmd_undergo_filtered.drop(columns=['ID', 'CLNSIG', 'CLNVC', 'GENEINFO', 'CLNREVSTAT', 'MC', 'SYMBOL', 
#                                                 'Gene', 'Feature_type', 'BIOTYPE', 'CANONICAL'])
ben_nmd_undergo_filtered = ben_nmd_undergo_filtered.drop(columns=['CLNSIG', 'CLNVC', 'GENEINFO', 'MC', 'CLNREVSTAT', 'Feature_type'])
ben_nmd_undergo_filtered

,CHROM,POS,ID,REF,ALT,AC,Consequence,IMPACT,SYMBOL,Gene,...,FLAGS,VARIANT_CLASS,CANONICAL,LoF,LoF_filter,LoF_flags,LoF_info,LOEUF,pext,NMD_escape
0,chr1,2025660,NaN,C,A,2.0,stop_gained,HIGH,GABRD,ENSG00000187730,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.288447387785136,0.245,1.000000,NO
1,chr1,2028164,rs1441225021,C,G,3.0,stop_gained,HIGH,GABRD,ENSG00000187730,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.414275202354673,0.245,1.000000,NO
2,chr1,2029236,rs1405824159,C,T,5.0,stop_gained,HIGH,GABRD,ENSG00000187730,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.601177336276674,0.245,1.000000,NO
3,chr1,2228818,rs1225111926,C,T,2.0,stop_gained,HIGH,SKI,ENSG00000157933,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.0237768632830361,0.194,0.950938,NO
4,chr1,2509920,rs1424097228,G,A,3.0,stop_gained,HIGH,PANK4,ENSG00000157881,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.882859603789836,0.304,0.694764,NO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5243,chr22,50720238,NaN,C,A,3.0,stop_gained,HIGH,SHANK3,ENSG00000251322,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.415837479270315,0.123,1.000000,NO
5244,chr22,50720534,NaN,G,T,2.0,stop_gained,HIGH,SHANK3,ENSG00000251322,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.477197346600332,0.123,1.000000,NO
5245,chr22,50721521,NaN,G,T,5.0,stop_gained,HIGH,SHANK3,ENSG00000251322,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.681799336650083,0.123,1.000000,NO
5246,chr22,50721995,NaN,G,T,2.0,stop_gained,HIGH,SHANK3,ENSG00000251322,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.780058043117745,0.123,1.000000,NO


In [23]:
ben_nmd_undergo_filtered.columns == pat_nmd_undergo.columns

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True])

In [24]:
ben_nmd_undergo_filtered.shape

(4909, 31)

Before filtering:

In [25]:
ben_nmd_undergo.shape

(5248, 31)

**The dataframe with benign variants is ready.**

In [26]:
pat_nmd_undergo_filtered = pat_nmd_undergo.copy()

**The dataframe with pathogenic variants is ready.**

### Add 5th group - Clinvar Pathogenic

In [27]:
clinvar_undergo_df = pd.read_csv("data/clinvar_nmd_undergo_df.csv")

In [28]:
clinvar_undergo_df.columns

Index(['CHROM', 'POS', 'ID', 'REF', 'ALT', 'CLNREVSTAT', 'CLNSIG', 'CLNVC',
       'GENEINFO', 'MC', 'Consequence', 'SYMBOL', 'Gene', 'Feature_type',
       'Feature', 'BIOTYPE', 'cDNA_position', 'CDS_position',
       'Protein_position', 'Amino_acids', 'Codons', 'STRAND', 'FLAGS',
       'CANONICAL', 'LOEUF', 'pext', 'NMD_escape'],
      dtype='object')

In [29]:
clinvar_undergo_df['Gene'].value_counts()

Gene
ENSG00000196712    697
ENSG00000095002    263
ENSG00000171316    196
ENSG00000100697    176
ENSG00000134982    170
                  ... 
ENSG00000182866      1
ENSG00000100345      1
ENSG00000004487      1
ENSG00000130940      1
ENSG00000078900      1
Name: count, Length: 445, dtype: int64

## 2. Balance dataframes

In both dataframes, we will leave only those genes that are found in both `ben_nmd_undergo_filtered` and `pat_nmd_undergo_filtered`, and also equalize the number of variants in each gene.

In [33]:
unique_genes_pat = np.array(unique_genes_pat, dtype=str)
unique_genes_ben = np.array(unique_genes_ben, dtype=str)
unique_genes_clinvar = np.array(unique_genes_clinvar, dtype=str)

# Находим пересечение
intersected_genes = np.intersect1d(np.intersect1d(unique_genes_pat, unique_genes_ben), unique_genes_clinvar)

print("Intersected genes:", len(intersected_genes))


Intersected genes: 256


In [34]:
common_genes = set(unique_genes_ben) & set(unique_genes_pat) & set(unique_genes_clinvar)

ben_nmd_undergo_filtered = ben_nmd_undergo_filtered[ben_nmd_undergo_filtered['SYMBOL'].isin(common_genes)]
pat_nmd_undergo_filtered = pat_nmd_undergo_filtered[pat_nmd_undergo_filtered['SYMBOL'].isin(common_genes)]
clinvar_undergo_df = clinvar_undergo_df[clinvar_undergo_df['SYMBOL'].isin(common_genes)]

In [35]:
# Проверка
unique_genes_ben = ben_nmd_undergo_filtered['SYMBOL'].unique()
unique_genes_pat = pat_nmd_undergo_filtered['SYMBOL'].unique()
unique_genes_clinvar = clinvar_undergo_df['SYMBOL'].unique()

intersected_genes = np.intersect1d(np.intersect1d(unique_genes_pat, unique_genes_ben), unique_genes_clinvar)

print("Unique genes in pat dataframe:", len(unique_genes_pat))
print("Unique genes in ben dataframe:", len(unique_genes_ben))
print("Unique genes in clinvar dataframe:", len(unique_genes_clinvar))
print("Intersected genes:", len(intersected_genes))

Unique genes in pat dataframe: 256
Unique genes in ben dataframe: 256
Unique genes in clinvar dataframe: 256
Intersected genes: 256


In [36]:
count_pat = pat_nmd_undergo_filtered['SYMBOL'].value_counts()
count_ben = ben_nmd_undergo_filtered['SYMBOL'].value_counts()
count_clinvar = clinvar_undergo_df['SYMBOL'].value_counts()

min_counts = pd.concat([count_pat, count_ben, count_clinvar], axis=1).min(axis=1)

pat_nmd_undergo_final = pd.concat([
    pat_nmd_undergo_filtered[pat_nmd_undergo_filtered['SYMBOL'] == gene].sample(n=min_count, random_state=42) \
    for gene, min_count in min_counts.items()
])

ben_nmd_undergo_final = pd.concat([
    ben_nmd_undergo_filtered[ben_nmd_undergo_filtered['SYMBOL'] == gene].sample(n=min_count, random_state=42) \
    for gene, min_count in min_counts.items()
])

clinvar_undergo_df = pd.concat([
    clinvar_undergo_df[clinvar_undergo_df['SYMBOL'] == gene].sample(n=min_count, random_state=42) \
    for gene, min_count in min_counts.items()
])

In [37]:
pat_nmd_undergo_final.shape == ben_nmd_undergo_final.shape

True

In [38]:
pat_nmd_undergo_final.shape

(532, 31)

In [39]:
ben_nmd_undergo_final.shape

(532, 31)

In [40]:
pat_nmd_undergo_final['SYMBOL'].nunique() == ben_nmd_undergo_final['SYMBOL'].nunique()

True

In [41]:
pat_nmd_undergo_final['SYMBOL'].value_counts().sum() == ben_nmd_undergo_final['SYMBOL'].value_counts().sum()

np.True_

In [42]:
pat_nmd_undergo_final['SYMBOL'].nunique()

256

In [43]:
ben_nmd_undergo_final['SYMBOL'].nunique()

256

In [44]:
pat_nmd_undergo_final.shape

(532, 31)

In [45]:
ben_nmd_undergo_final.shape

(532, 31)

Dataframes are now balanced by genes and number of variants.

## 3. Get sequence context

Write the context in the corresponding column of the dataframe.

File `gencode.v47.transcripts.fa.gz` can be downloaded from the GENCODE database (https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_47/gencode.v47.transcripts.fa.gz).

In [46]:
ben_nmd_undergo_final.dtypes

CHROM                object
POS                   int64
ID                   object
REF                  object
ALT                  object
AC                  float64
Consequence          object
IMPACT               object
SYMBOL               object
Gene                 object
Feature              object
BIOTYPE              object
EXON                 object
INTRON              float64
cDNA_position       float64
CDS_position        float64
Protein_position    float64
Amino_acids          object
Codons               object
ALLELE_NUM          float64
STRAND              float64
FLAGS               float64
VARIANT_CLASS        object
CANONICAL            object
LoF                  object
LoF_filter          float64
LoF_flags           float64
LoF_info             object
LOEUF               float64
pext                float64
NMD_escape           object
dtype: object

In [47]:
ben_nmd_undergo_final['Codons']

1668    Cag/Tag
1669    Cga/Tga
1673    Cga/Tga
1670    Cag/Tag
1672    tTa/tAa
         ...   
3389    Cag/Tag
5082    tGg/tAg
1099    Gag/Tag
3463    Cga/Tga
2910    Gag/Tag
Name: Codons, Length: 532, dtype: object

In [48]:
ben_nmd_undergo_final['cDNA_position'] = ben_nmd_undergo_final['cDNA_position'].astype(int)

In [49]:
pat_nmd_undergo_final['cDNA_position'] = pat_nmd_undergo_final['cDNA_position'].astype(int)

In [50]:
clinvar_undergo_df['cDNA_position'] = clinvar_undergo_df['cDNA_position'].astype(int)

In [51]:
transcript_fasta = Fasta("data/gencode.v47.transcripts.fa.gz", key_function = lambda x: x.split('.')[0])

In [52]:
get_context(ben_nmd_undergo_final, transcript_fasta, 13, 12)

'Contexts have been added to the dataframe!'

In [53]:
get_context(pat_nmd_undergo_final, transcript_fasta, 13, 12)

'Contexts have been added to the dataframe!'

In [54]:
get_context(clinvar_undergo_df, transcript_fasta, 13, 12)

'Contexts have been added to the dataframe!'

In [55]:
pat_nmd_undergo_final.iloc[1]

CHROM                                       chr5
POS                                    128408791
ID                                           NaN
REF                                            C
ALT                                            A
AC                                             1
Consequence                          stop_gained
IMPACT                                      HIGH
SYMBOL                                      FBN2
Gene                             ENSG00000138829
Feature                          ENST00000262464
BIOTYPE                           protein_coding
EXON                                        8/65
INTRON                                       NaN
cDNA_position                               1603
CDS_position                               961.0
Protein_position                           321.0
Amino_acids                                  E/*
Codons                                   Gag/Tag
ALLELE_NUM                                     1
STRAND              

## 4. Get variant codons information

In [56]:
ben_nmd_undergo_final['Codons']

1668    Cag/Tag
1669    Cga/Tga
1673    Cga/Tga
1670    Cag/Tag
1672    tTa/tAa
         ...   
3389    Cag/Tag
5082    tGg/tAg
1099    Gag/Tag
3463    Cga/Tga
2910    Gag/Tag
Name: Codons, Length: 532, dtype: object

In [57]:
def get_codon_position(codon_change):
    codon = codon_change.split('/')[0]  # первый кодон (до слэша)
    for i, base in enumerate(codon, start=1):
        if base.isupper():
            return i
    return None  # если вдруг нет заглавной буквы

In [58]:
ben_nmd_undergo_final['Codon_position'] = ben_nmd_undergo_final['Codons'].apply(get_codon_position)

In [59]:
ben_nmd_undergo_final['Codon_position'].value_counts()

Codon_position
1    394
2     72
3     66
Name: count, dtype: int64

In [60]:
pat_nmd_undergo_final['Codon_position'] = pat_nmd_undergo_final['Codons'].apply(get_codon_position)

In [61]:
pat_nmd_undergo_final['Codon_position'].value_counts()

Codon_position
1    322
3    124
2     86
Name: count, dtype: int64

In [62]:
clinvar_undergo_df['Codon_position'] = clinvar_undergo_df['Codons'].apply(get_codon_position)

In [63]:
clinvar_undergo_df['Codon_position'].value_counts()

Codon_position
1    382
3     84
2     66
Name: count, dtype: int64

In [64]:
def get_initial_codon(codon_change):
    return codon_change.split('/')[0].upper()

def get_stop_codon(codon_change):
    return codon_change.split('/')[1].upper()

In [65]:
ben_nmd_undergo_final['Initial_Codon'] = ben_nmd_undergo_final['Codons'].apply(get_initial_codon)
ben_nmd_undergo_final['Stop_Codon'] = ben_nmd_undergo_final['Codons'].apply(get_stop_codon)

In [66]:
pat_nmd_undergo_final['Initial_Codon'] = pat_nmd_undergo_final['Codons'].apply(get_initial_codon)
pat_nmd_undergo_final['Stop_Codon'] = pat_nmd_undergo_final['Codons'].apply(get_stop_codon)

In [67]:
clinvar_undergo_df['Initial_Codon'] = clinvar_undergo_df['Codons'].apply(get_initial_codon)
clinvar_undergo_df['Stop_Codon'] = clinvar_undergo_df['Codons'].apply(get_stop_codon)

In [68]:
ben_nmd_undergo_final['Codon_change'] = list(zip(ben_nmd_undergo_final['Initial_Codon'], ben_nmd_undergo_final['Stop_Codon']))
ben_nmd_undergo_final['Codon_change'].value_counts(normalize=True) * 100

Codon_change
(CGA, TGA)    39.285714
(CAG, TAG)    13.533835
(GAG, TAG)     9.022556
(GAA, TAA)     5.075188
(TCA, TAA)     4.699248
(TGG, TAG)     3.947368
(CAA, TAA)     3.007519
(TAC, TAA)     3.007519
(TGG, TGA)     3.007519
(TAC, TAG)     2.631579
(GGA, TGA)     2.067669
(TCG, TAG)     1.879699
(TGC, TGA)     1.879699
(TAT, TAG)     1.503759
(TCA, TGA)     1.315789
(AAG, TAG)     1.127820
(AAA, TAA)     0.751880
(TTA, TAA)     0.751880
(TTA, TGA)     0.751880
(AGA, TGA)     0.187970
(TTG, TAG)     0.187970
(TAT, TAA)     0.187970
(TGT, TGA)     0.187970
Name: proportion, dtype: float64

In [69]:
pat_nmd_undergo_final['Codon_change'] = list(zip(pat_nmd_undergo_final['Initial_Codon'], pat_nmd_undergo_final['Stop_Codon']))
pat_nmd_undergo_final['Codon_change'].value_counts(normalize=True) * 100

Codon_change
(CAG, TAG)    18.984962
(CGA, TGA)    13.721805
(GAG, TAG)     9.586466
(TAC, TAA)     5.827068
(CAA, TAA)     5.639098
(TGG, TGA)     5.639098
(TCA, TAA)     5.639098
(TAC, TAG)     4.323308
(GAA, TAA)     4.135338
(TGG, TAG)     3.759398
(GGA, TGA)     3.759398
(TGC, TGA)     3.383459
(TCA, TGA)     2.255639
(TCG, TAG)     2.255639
(AAG, TAG)     2.255639
(AAA, TAA)     1.691729
(TGT, TGA)     1.503759
(TAT, TAA)     1.315789
(TTA, TGA)     1.315789
(TAT, TAG)     1.315789
(AGA, TGA)     0.751880
(TTG, TAG)     0.751880
(TTA, TAA)     0.187970
Name: proportion, dtype: float64

In [70]:
clinvar_undergo_df['Codon_change'] = list(zip(clinvar_undergo_df['Initial_Codon'], clinvar_undergo_df['Stop_Codon']))
clinvar_undergo_df['Codon_change'].value_counts(normalize=True) * 100

Codon_change
(CGA, TGA)    30.827068
(CAG, TAG)    19.172932
(TGG, TAG)     6.203008
(GAG, TAG)     5.827068
(GAA, TAA)     5.827068
(CAA, TAA)     5.075188
(TGG, TGA)     3.759398
(TAC, TAG)     3.383459
(TAC, TAA)     3.007519
(TCA, TGA)     2.067669
(AAG, TAG)     2.067669
(TGC, TGA)     1.691729
(TAT, TAA)     1.691729
(TAT, TAG)     1.691729
(GGA, TGA)     1.503759
(TCA, TAA)     1.315789
(TCG, TAG)     1.127820
(AAA, TAA)     0.939850
(TTA, TGA)     0.751880
(TGT, TGA)     0.563910
(TTG, TAG)     0.563910
(AGA, TGA)     0.563910
(TTA, TAA)     0.375940
Name: proportion, dtype: float64

## 8. Relationship between the significance of a variant and its position in a codon

In [71]:
pat_nmd_undergo_final.loc[:, 'Significance'] = 'pathogenic'
ben_nmd_undergo_final.loc[:, 'Significance'] = 'benign'

In [72]:
all_nmd_undergo_final = pd.concat([pat_nmd_undergo_final, ben_nmd_undergo_final], ignore_index=True)

In [73]:
all_nmd_undergo_final = all_nmd_undergo_final.loc[all_nmd_undergo_final['Codon_position'] != 'No_stop']

In [74]:
all_nmd_undergo_final.to_csv('data/2A-i_all_nmd_undergo_final_wo_in_clinvar.csv', index=False)

In [75]:
clinvar_undergo_df.to_csv('data/2A-i_clinvar_nmd_undergo_final.csv', index=False)

In [79]:
clinvar_undergo_df

,CHROM,POS,ID,REF,ALT,CLNREVSTAT,CLNSIG,CLNVC,GENEINFO,MC,...,FLAGS,CANONICAL,LOEUF,pext,NMD_escape,Context,Codon_position,Initial_Codon,Stop_Codon,Codon_change
2053,chr5,128330622,1333458,G,T,"criteria_provided,_single_submitter",Pathogenic,single_nucleotide_variant,FBN2:2201,SO:0001587|nonsense,...,NaN,YES,0.205,0.993321,NO,CCCGGGCTCATACCGCTGTGCCTGC,3,TAC,TAA,"(TAC, TAA)"
2056,chr5,128537513,1709548,G,A,"criteria_provided,_single_submitter",Likely_pathogenic,single_nucleotide_variant,FBN2:2201,SO:0001587|nonsense,...,NaN,YES,0.205,0.841817,NO,GCCGGCCAGCCTCAGCCTCCTCCGC,1,CAG,TAG,"(CAG, TAG)"
2054,chr5,128334836,992339,C,A,"criteria_provided,_single_submitter",Likely_pathogenic,single_nucleotide_variant,FBN2:2201,SO:0001587|nonsense,...,NaN,YES,0.205,0.993321,NO,ATTGATGTCAATGAATGTGACCTAA,1,GAA,TAA,"(GAA, TAA)"
2052,chr5,128274648,225357,G,A,"criteria_provided,_single_submitter",Likely_pathogenic,single_nucleotide_variant,FBN2:2201,SO:0001587|nonsense,...,NaN,YES,0.205,0.839345,NO,CAGCATAACTGCCAGTTCCTCTGTG,1,CAG,TAG,"(CAG, TAG)"
2055,chr5,128527879,213262,A,T,"criteria_provided,_single_submitter",Pathogenic,single_nucleotide_variant,FBN2:2201,SO:0001587|nonsense,...,NaN,YES,0.205,0.841817,NO,TGGAACTTATTGTGGACAACCTGTC,3,TGT,TGA,"(TGT, TGA)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4015,chr11,86954968,1071058,C,A,"criteria_provided,_single_submitter",Pathogenic,single_nucleotide_variant,FZD4:8322,SO:0001587|nonsense,...,NaN,YES,0.312,1.000000,NO,GGCTTCGGGGACGAGGAAGAGCGGC,1,GAG,TAG,"(GAG, TAG)"
6900,chr20,63472179,449910,G,T,"criteria_provided,_multiple_submitters,_no_con...",Pathogenic/Likely_pathogenic,single_nucleotide_variant,KCNQ2:3785,SO:0001587|nonsense,...,NaN,YES,0.158,0.978148,NO,GGCGTTCATCTACCACGCCTACGTG,3,TAC,TAA,"(TAC, TAA)"
1449,chr3,128485729,1184235,G,T,"criteria_provided,_single_submitter",Pathogenic,single_nucleotide_variant,GATA2:2624,SO:0001587|nonsense,...,NaN,YES,0.292,0.695356,NO,CTCGTTCCTGTTCAGAAGGCCGGGA,2,TCA,TAA,"(TCA, TAA)"
4149,chr12,11884450,3343928,A,T,"criteria_provided,_single_submitter",Pathogenic,single_nucleotide_variant,ETV6:2120|LOC126861452:126861452,"SO:0001587|nonsense,SO:0001627|intron_variant",...,NaN,YES,0.318,0.944509,NO,ATAGCAGACTGTAGACTGCTTTGGG,1,AGA,TGA,"(AGA, TGA)"


In [80]:
pat_nmd_undergo_final

,CHROM,POS,ID,REF,ALT,AC,Consequence,IMPACT,SYMBOL,Gene,...,LoF_info,LOEUF,pext,NMD_escape,Context,Codon_position,Initial_Codon,Stop_Codon,Codon_change,Significance
6557,chr5,128527945,NaN,G,T,1,stop_gained,HIGH,FBN2,ENSG00000138829,...,PERCENTILE:0.0525231719876416,0.205,0.841817,NO,CAGTGTGAGATGCATGAATGGTGGG,3,TGC,TGA,"(TGC, TGA)",pathogenic
6553,chr5,128408791,NaN,C,A,1,stop_gained,HIGH,FBN2,ENSG00000138829,...,PERCENTILE:0.109966815425106,0.205,0.839350,NO,GAAGACATTGATGAGTGCAGCATCA,1,GAG,TAG,"(GAG, TAG)",pathogenic
6560,chr5,128536472,NaN,G,T,1,stop_gained,HIGH,FBN2,ENSG00000138829,...,PERCENTILE:0.0305526948163405,0.205,0.841817,NO,GCCCAACGTGTGCGGCTCCAGATTC,3,TGC,TGA,"(TGC, TGA)",pathogenic
6520,chr5,128304972,NaN,G,A,1,stop_gained,HIGH,FBN2,ENSG00000138829,...,PERCENTILE:0.661975054354045,0.205,0.839345,NO,GCTTCTCAGGACCAGACCATGTGCA,1,CAG,TAG,"(CAG, TAG)",pathogenic
6533,chr5,128334820,NaN,G,C,1,stop_gained,HIGH,FBN2,ENSG00000138829,...,PERCENTILE:0.457489415264904,0.205,0.993321,NO,GTGACCTAAATTCAAATATCTGCAT,2,TCA,TGA,"(TCA, TGA)",pathogenic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13002,chr11,86954965,NaN,C,A,1,stop_gained,HIGH,FZD4,ENSG00000174804,...,PERCENTILE:0.0749690210656753,0.312,1.000000,NO,TTCGGGGACGAGGAAGAGCGGCGCT,1,GAA,TAA,"(GAA, TAA)",pathogenic
20327,chr20,63444706,NaN,C,A,1,stop_gained,HIGH,KCNQ2,ENSG00000075043,...,PERCENTILE:0.245513554791905,0.158,0.978148,NO,ATGGACCGGCGGGGAGGCACCTGGA,1,GGA,TGA,"(GGA, TGA)",pathogenic
4306,chr3,128483889,rs1576746847,G,A,1,stop_gained,HIGH,GATA2,ENSG00000179348,...,PERCENTILE:0.684684684684685,0.292,0.695356,NO,AATGGGCAGAACCGACCACTCATCA,1,CGA,TGA,"(CGA, TGA)",pathogenic
13360,chr12,11853525,NaN,C,T,1,stop_gained,HIGH,ETV6,ENSG00000139083,...,PERCENTILE:0.31420161883738,0.318,0.944509,NO,TCTATACACACACAGCCGGAGGTCA,1,CAG,TAG,"(CAG, TAG)",pathogenic


In [81]:
ben_nmd_undergo_final

,CHROM,POS,ID,REF,ALT,AC,Consequence,IMPACT,SYMBOL,Gene,...,LoF_info,LOEUF,pext,NMD_escape,Context,Codon_position,Initial_Codon,Stop_Codon,Codon_change,Significance
1668,chr5,128289939,rs1240185443,G,A,2.0,stop_gained,HIGH,FBN2,ENSG00000138829,...,PERCENTILE:0.738528435747797,0.205,0.839345,NO,GAAGTTGCATTTCAGGATTTGTGTC,1,CAG,TAG,"(CAG, TAG)",benign
1669,chr5,128309376,rs1178371653,G,A,2.0,stop_gained,HIGH,FBN2,ENSG00000138829,...,PERCENTILE:0.59778006636915,0.205,0.839345,NO,AGCTTTTGCTACCGAAGCTATAATG,1,CGA,TGA,"(CGA, TGA)",benign
1673,chr5,128408683,rs1752991946,G,A,3.0,stop_gained,HIGH,FBN2,ENSG00000138829,...,PERCENTILE:0.122325208833963,0.205,0.839350,NO,ACAGATGGCTCTCGATGCATCGATC,1,CGA,TGA,"(CGA, TGA)",benign
1670,chr5,128336042,rs863223569,G,A,4.0,stop_gained,HIGH,FBN2,ENSG00000138829,...,PERCENTILE:0.419956516763932,0.205,0.993321,NO,ATTGGAACCTATCAGTGCTCTTGCA,1,CAG,TAG,"(CAG, TAG)",benign
1672,chr5,128361833,NaN,A,T,2.0,stop_gained,HIGH,FBN2,ENSG00000138829,...,PERCENTILE:0.279665865659686,0.205,0.839350,NO,TTGATGAATGTTTAGTAAACAGACT,2,TTA,TAA,"(TTA, TAA)",benign
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3389,chr11,86954872,NaN,G,A,2.0,stop_gained,HIGH,FZD4,ENSG00000174804,...,PERCENTILE:0.132589838909542,0.312,1.000000,NO,GGGCACGAGCTGCAGACGGACGCCG,1,CAG,TAG,"(CAG, TAG)",benign
5082,chr20,63445282,NaN,C,T,2.0,stop_gained,HIGH,KCNQ2,ENSG00000075043,...,PERCENTILE:0.179457808323788,0.158,0.978148,NO,GGTACCGTGGCTGGAGGGGGCGGCT,2,TGG,TAG,"(TGG, TAG)",benign
1099,chr3,128487016,NaN,C,A,2.0,stop_gained,HIGH,GATA2,ENSG00000179348,...,PERCENTILE:0.0110880110880111,0.292,0.700120,NO,GAGGTGGCGCCCGAGCAGCCGCGCT,1,GAG,TAG,"(GAG, TAG)",benign
3463,chr12,11752531,rs1036410712,C,T,5.0,stop_gained,HIGH,ETV6,ENSG00000139083,...,PERCENTILE:0.0846210448859455,0.318,0.976427,NO,GTTCCAGTGCCTCGAGCGCTCAGGA,1,CGA,TGA,"(CGA, TGA)",benign
